# OGI Kids ASR — final notebook with preflight checks and stabilized Wav2Vec2

This notebook is designed to be run top to bottom in a fresh Colab runtime.

What it includes:
- **PASS / FAIL preflight checks**
- **path-based audio loading with `soundfile`**
- **Whisper task runs**
- **stabilized Wav2Vec2 fine-tuning**
- **automatic best-config selection after Task 3**
- **autosaving results to Google Drive after each major stage**

Wav2Vec2 stabilization used here:
- frozen feature encoder
- original simple CTC filter
- `ctc_zero_infinity=True`
- `max_grad_norm=0.5`
- disable train-time SpecAugment masking:
  - `apply_spec_augment = False`
  - `mask_time_prob = 0.0`
  - `mask_feature_prob = 0.0`
- freeze `wav2vec2.masked_spec_embed`

In [1]:

import sys
!{sys.executable} -m pip install -q --no-cache-dir \
    "transformers>=4.51,<4.53" \
    "tokenizers>=0.21,<0.22" \
    "datasets>=3.5,<4" \
    "accelerate>=1.6,<2" \
    "soundfile>=0.12.1" \
    "evaluate>=0.4,<0.5" \
    "jiwer>=3,<4" \
    "torchaudio"

print("Install cell finished. If your environment was messy, restart runtime once before continuing.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 180.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 292.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 427.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 350.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 428.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 420.0 MB/s eta 0:00:00
Install cell finished. If your environment was messy, restart runtime once before continuing.


In [2]:

import os
import zipfile

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

DATA_ZIP_PATH = "/content/drive/MyDrive/S26_ECE_214B_Mini_Project_1.zip"
EXTRACT_ROOT = "/content"
PROJECT_ROOT = os.path.join(EXTRACT_ROOT, "S26_ECE_214B_Mini_Project_1")

if not os.path.exists(DATA_ZIP_PATH):
    raise FileNotFoundError(f"Could not find dataset zip at: {DATA_ZIP_PATH}")

os.makedirs(EXTRACT_ROOT, exist_ok=True)
with zipfile.ZipFile(DATA_ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_ROOT)

print("Extracted to:", PROJECT_ROOT)
print("Exists:", os.path.exists(PROJECT_ROOT))

Mounted at /content/drive
Extracted to: /content/S26_ECE_214B_Mini_Project_1
Exists: False


In [8]:

import os
import re
import gc
import math
import shutil
import random
import warnings
from typing import Any, Dict, List, Union

import numpy as np
import pandas as pd
import torch
import soundfile as sf

from datasets import Dataset, DatasetDict, concatenate_datasets
from transformers import (
    set_seed,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
)
import evaluate

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
wer_metric = evaluate.load("wer")

print("device =", device)

def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def make_fresh_output_dir(path: str):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def check(name: str, condition: bool, detail: str = "", hard: bool = False):
    status = "PASS" if condition else "FAIL"
    msg = f"[{status}] {name}"
    if detail:
        msg += f" :: {detail}"
    print(msg)
    if hard and not condition:
        raise AssertionError(msg)

results = []

def add_result(
    family,
    model_name,
    setting,
    learning_rate,
    batch_size,
    max_steps,
    train_size,
    train_age_group,
    test_age_group,
    valid_wer,
    test_wer,
    augmentation="none",
    notes=""
):
    results.append({
        "family": family,
        "model_name": model_name,
        "setting": setting,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "max_steps": max_steps,
        "train_size": train_size,
        "train_age_group": train_age_group,
        "test_age_group": test_age_group,
        "augmentation": augmentation,
        "valid_wer": valid_wer,
        "test_wer": test_wer,
        "notes": notes,
    })

def results_df():
    if len(results) == 0:
        return pd.DataFrame()
    return (
        pd.DataFrame(results)
        .sort_values(by=["family", "test_wer", "valid_wer"], ascending=[True, True, True])
        .reset_index(drop=True)
    )

AUTO_RESULTS_PATH = "/content/drive/MyDrive/ogi_kids_asr_results_autosave.csv"

def save_results_snapshot():
    if len(results) == 0:
        print("No results yet; nothing to save.")
        return
    df = results_df()
    try:
        df.to_csv(AUTO_RESULTS_PATH, index=False)
        print("Autosaved results to:", AUTO_RESULTS_PATH)
    except Exception as e:
        print("Autosave skipped:", repr(e))

def maybe_augment_audio(arr, p_noise=0.5, p_gain=0.5):
    x = arr.astype(np.float32).copy()
    if np.random.rand() < p_noise:
        noise = np.random.randn(len(x)).astype(np.float32) * 0.003
        x = x + noise
    if np.random.rand() < p_gain:
        gain = np.random.uniform(0.8, 1.2)
        x = x * gain
    x = np.clip(x, -1.0, 1.0)
    return x.astype(np.float32)

def normalize_w2v_text(text):
    text = text.upper()
    text = re.sub(r"[^A-Z' ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

WAV2VEC2_DOWNSAMPLE = 320

def valid_ctc_sample_original(example):
    return (
        example["label_length"] > 0
        and (example["input_length"] // WAV2VEC2_DOWNSAMPLE) > example["label_length"]
    )

device = cuda


In [4]:
! cp /content/drive/MyDrive/S26_ECE214B_MiniProject1_BlindTest.zip /content

In [5]:
! unzip /content/S26_ECE214B_MiniProject1_BlindTest.zip -d /content/

Archive:  /content/S26_ECE214B_MiniProject1_BlindTest.zip
   creating: /content/speech/
   creating: /content/speech/speech/
   creating: /content/speech/speech/scripted/
   creating: /content/speech/speech/scripted/00/
   creating: /content/speech/speech/scripted/01/
   creating: /content/speech/speech/scripted/02/
   creating: /content/speech/speech/scripted/03/
   creating: /content/speech/speech/scripted/04/
   creating: /content/speech/speech/scripted/05/
   creating: /content/speech/speech/scripted/06/
   creating: /content/speech/speech/scripted/07/
   creating: /content/speech/speech/scripted/08/
   creating: /content/speech/speech/scripted/09/
   creating: /content/speech/speech/scripted/10/
   creating: /content/speech/speech/scripted/06/0/
   creating: /content/speech/speech/scripted/06/1/
   creating: /content/speech/speech/scripted/06/2/
   creating: /content/speech/speech/scripted/06/3/
   creating: /content/speech/speech/scripted/06/4/
   creating: /content/speech/speech

In [11]:

base_path = '/content/data/'
ogi_train_path = {
    "data": {"scp_path": f"{base_path}/train/wav.scp", "text_label": f"{base_path}/train/text"}
}
ogi_dev_path = {
    "data": {"scp_path": f"{base_path}/dev/wav.scp", "text_label": f"{base_path}/dev/text"}
}
ogi_test_path = {
    "data": {"scp_path": f"{base_path}/test/wav.scp", "text_label": f"{base_path}/test/text"}
}
ogi_blind_path = {
    "data": {"scp_path": f"{base_path}/blind/wav.scp", "text_label": f"{base_path}/blind/text"}
}

class OGIMetadataDataset:
    def __init__(self, data_paths, sample_limit=None, age_range=None, seed=42):
        self.data_paths = data_paths
        self.sample_limit = sample_limit
        self.age_range = age_range
        self.seed = seed
        datasets = []
        for split in data_paths.keys():
            datasets.append(self.create_dataset(data_paths[split]))
        self.data = concatenate_datasets(datasets)

    def create_dataset(self, data_path):
        audio_dict = self._load_audio_path(data_path["scp_path"])
        label_dict = self._load_label(data_path["text_label"])
        assert len(audio_dict) == len(label_dict), "label and sample size mismatch"

        paired_dict = {"utt_id": [], "audio_path": [], "sentence": [], "grade": [], "age": []}

        for utt, audio_path in audio_dict:
            label = label_dict[utt]
            grade_str = audio_path.split("/")[-4]
            grade = int(grade_str)
            age = grade + 4

            if self.age_range is not None:
                lo, hi = self.age_range
                if not (lo <= age <= hi):
                    continue

            paired_dict["utt_id"].append(utt)
            paired_dict["audio_path"].append(audio_path)
            paired_dict["sentence"].append(label)
            paired_dict["grade"].append(grade)
            paired_dict["age"].append(age)

        dataset = Dataset.from_dict(paired_dict)
        if self.sample_limit is not None:
            n = min(self.sample_limit, len(dataset))
            dataset = dataset.shuffle(seed=self.seed).select(range(n))
        return dataset

    def _load_audio_path(self, wav_path):
        audio_dict = []
        with open(wav_path, "r") as fin:
            line = fin.readline()
            while line:
                line = line.strip().split(" ")
                utt = line[0]
                path = line[-1].replace("./", f"{PROJECT_ROOT}/OGI_Kids/")
                audio_dict.append((utt, path))
                line = fin.readline()
        print(f"Reading {len(audio_dict)} lines from {wav_path}")
        return audio_dict

    def _load_label(self, lab_path):
        label_dict = {}
        with open(lab_path, "r") as fin:
            line = fin.readline()
            while line:
                line = line.strip().split(" ")
                label_dict[line[0]] = " ".join(line[1:])
                line = fin.readline()
        print(f"Reading {len(label_dict)} lines from {lab_path}")
        return label_dict

def build_ogi_splits(train_size=None, train_age_range=None, test_age_range=None, seed=42):
    train = OGIMetadataDataset(ogi_train_path, sample_limit=train_size, age_range=train_age_range, seed=seed).data
    valid = OGIMetadataDataset(ogi_dev_path, age_range=train_age_range, seed=seed).data
    test = OGIMetadataDataset(ogi_test_path, age_range=test_age_range, seed=seed).data
    return DatasetDict({"train": train, "valid": valid, "test": test})

splits = build_ogi_splits()
print(splits)
print("train:", len(splits["train"]), "valid:", len(splits["valid"]), "test:", len(splits["test"]))
print("sample transcript:", splits["train"]["sentence"][0])
print("sample age:", splits["train"]["age"][0], "grade:", splits["train"]["grade"][0])

check("Dataset counts", len(splits["train"]) == 20000 and len(splits["valid"]) == 2000 and len(splits["test"]) == 2000,
      detail=f"{len(splits['train'])}/{len(splits['valid'])}/{len(splits['test'])}", hard=True)

Reading 20000 lines from /content/data//train/wav.scp
Reading 20000 lines from /content/data//train/text
Reading 2000 lines from /content/data//dev/wav.scp
Reading 2000 lines from /content/data//dev/text
Reading 2000 lines from /content/data//test/wav.scp
Reading 2000 lines from /content/data//test/text
DatasetDict({
    train: Dataset({
        features: ['utt_id', 'audio_path', 'sentence', 'grade', 'age'],
        num_rows: 20000
    })
    valid: Dataset({
        features: ['utt_id', 'audio_path', 'sentence', 'grade', 'age'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['utt_id', 'audio_path', 'sentence', 'grade', 'age'],
        num_rows: 2000
    })
})
train: 20000 valid: 2000 test: 2000
sample transcript: ADVANTAGE
sample age: 4 grade: 0
[PASS] Dataset counts :: 20000/2000/2000


In [14]:

for idx in [0, 1, 2]:
    p = splits["train"]["audio_path"][idx]
    print(idx, p, "exists =", os.path.exists(p))
    assert os.path.exists(p), f"Audio path missing: {p}"
check("A1 audio paths exist", True, hard=True)

0 /content/S26_ECE_214B_Mini_Project_1/OGI_Kids/speech/scripted/00/0/ks001/ks001000.wav exists = False


AssertionError: Audio path missing: /content/S26_ECE_214B_Mini_Project_1/OGI_Kids/speech/scripted/00/0/ks001/ks001000.wav

In [13]:

sample_rows = []
for idx in [0, 25, 100]:
    path = splits["train"]["audio_path"][idx]
    sentence = splits["train"]["sentence"][idx]
    info = sf.info(path)
    sample_rows.append({
        "idx": idx,
        "path": path,
        "exists": os.path.exists(path),
        "sr": info.samplerate,
        "frames": info.frames,
        "duration_sec": round(info.frames / info.samplerate, 3),
        "sentence": sentence,
    })
df_a2 = pd.DataFrame(sample_rows)
display(df_a2)
ok = bool(df_a2["exists"].all() and (df_a2["sr"] == 16000).all() and (df_a2["duration_sec"] > 0).all())
check("A2 soundfile metadata check", ok, hard=True)

LibsndfileError: Error opening '/content/S26_ECE_214B_Mini_Project_1/OGI_Kids/speech/scripted/00/0/ks001/ks001000.wav': System error.

In [ ]:

whisper_proc_debug = WhisperProcessor.from_pretrained("openai/whisper-base.en")
whisper_input_name = whisper_proc_debug.feature_extractor.model_input_names[0]
def inspect_whisper_sample_from_path(path, sentence):
    audio_arr, sr = sf.read(path)
    audio_arr = audio_arr.astype(np.float32)
    inputs = whisper_proc_debug.feature_extractor(audio_arr, sampling_rate=sr)
    return {
        "path": path,
        "sampling_rate": sr,
        "num_samples": len(audio_arr),
        "input_feature_shape": np.array(inputs.get(whisper_input_name)[0]).shape,
        "sentence": sentence,
    }
rows = [inspect_whisper_sample_from_path(splits["train"]["audio_path"][i], splits["train"]["sentence"][i]) for i in [0,1,2]]
display(pd.DataFrame(rows))
check("A4 Whisper preprocessing sanity", all(r["sampling_rate"] == 16000 for r in rows) and all(r["input_feature_shape"][0] == 80 for r in rows), hard=True)

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

,path,sampling_rate,num_samples,input_feature_shape,sentence
0,/content/S26_ECE_214B_Mini_Project_1/OGI_Kids/...,16000,65472,"(80, 3000)",ADVANTAGE
1,/content/S26_ECE_214B_Mini_Project_1/OGI_Kids/...,16000,73585,"(80, 3000)",BEHIND
2,/content/S26_ECE_214B_Mini_Project_1/OGI_Kids/...,16000,41842,"(80, 3000)",BOOMERANG


[PASS] A4 Whisper preprocessing sanity


In [ ]:

w2v_proc_debug = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
def inspect_w2v_sample_from_path(path, sentence):
    audio_arr, sr = sf.read(path)
    audio_arr = audio_arr.astype(np.float32)
    input_values = w2v_proc_debug(audio_arr, sampling_rate=sr).input_values[0]
    labels = w2v_proc_debug.tokenizer(normalize_w2v_text(sentence)).input_ids
    return {
        "path": path,
        "sampling_rate": sr,
        "num_samples": len(audio_arr),
        "input_values_len": len(input_values),
        "label_len": len(labels),
        "normalized_text": normalize_w2v_text(sentence),
    }
rows = [inspect_w2v_sample_from_path(splits["train"]["audio_path"][i], splits["train"]["sentence"][i]) for i in [0,1,2]]
display(pd.DataFrame(rows))
check("A5 W2V preprocessing sanity", all(r["sampling_rate"] == 16000 for r in rows) and all(r["input_values_len"] > 0 for r in rows), hard=True)

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

,path,sampling_rate,num_samples,input_values_len,label_len,normalized_text
0,/content/S26_ECE_214B_Mini_Project_1/OGI_Kids/...,16000,65472,65472,9,ADVANTAGE
1,/content/S26_ECE_214B_Mini_Project_1/OGI_Kids/...,16000,73585,73585,6,BEHIND
2,/content/S26_ECE_214B_Mini_Project_1/OGI_Kids/...,16000,41842,41842,9,BOOMERANG


[PASS] A5 W2V preprocessing sanity


In [ ]:

class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor: Any, decoder_start_token_id: int):
        self.processor = processor
        self.decoder_start_token_id = decoder_start_token_id
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

class DataCollatorCTCWithPadding:
    def __init__(self, processor, padding=True):
        self.processor = processor
        self.padding = padding
    def __call__(self, features):
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

check("Collator definitions", True)

[PASS] Collator definitions


In [ ]:

whisper_proc = WhisperProcessor.from_pretrained("openai/whisper-base.en")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base.en")
whisper_input_name = whisper_proc.feature_extractor.model_input_names[0]
def whisper_prepare_from_path(path, sentence):
    audio_arr, sr = sf.read(path)
    audio_arr = audio_arr.astype(np.float32)
    inputs = whisper_proc.feature_extractor(audio_arr, sampling_rate=sr)
    labels = whisper_proc.tokenizer(whisper_proc.tokenizer.normalize(sentence)).input_ids
    return {whisper_input_name: inputs.get(whisper_input_name)[0], "labels": labels}
whisper_debug_features = [whisper_prepare_from_path(splits["train"]["audio_path"][i], splits["train"]["sentence"][i]) for i in range(4)]
whisper_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=whisper_proc, decoder_start_token_id=whisper_model.config.decoder_start_token_id)
wbatch = whisper_collator(whisper_debug_features)
print("Whisper batch shapes:")
for k, v in wbatch.items():
    print(k, tuple(v.shape))
check("A6.1 Whisper collator", ("input_features" in wbatch and "labels" in wbatch and wbatch["input_features"].shape[0] == 4), hard=True)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

Whisper batch shapes:
input_features (4, 80, 3000)
labels (4, 5)
[PASS] A6.1 Whisper collator


In [ ]:

w2v_proc = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
def w2v_prepare_from_path(path, sentence):
    audio_arr, sr = sf.read(path)
    audio_arr = audio_arr.astype(np.float32)
    input_values = w2v_proc(audio_arr, sampling_rate=sr).input_values[0]
    labels = w2v_proc.tokenizer(normalize_w2v_text(sentence)).input_ids
    return {"input_values": input_values, "labels": labels, "input_length": len(input_values), "label_length": len(labels)}
w2v_debug_features = []
for i in range(16):
    feat = w2v_prepare_from_path(splits["train"]["audio_path"][i], splits["train"]["sentence"][i])
    if valid_ctc_sample_original(feat):
        w2v_debug_features.append(feat)
w2v_collator = DataCollatorCTCWithPadding(processor=w2v_proc, padding=True)
cbatch = w2v_collator(w2v_debug_features[:4])
print("W2V batch shapes:")
for k, v in cbatch.items():
    print(k, tuple(v.shape))
print("Any all--100 label rows?", bool(((cbatch["labels"] == -100).all(dim=1)).any().item()))
check("A6.2 W2V collator", ("input_values" in cbatch and "labels" in cbatch and not bool(((cbatch["labels"] == -100).all(dim=1)).any().item())), hard=True)

W2V batch shapes:
input_values (4, 73585)
labels (4, 9)
Any all--100 label rows? False
[PASS] A6.2 W2V collator


In [ ]:

def whisper_zero_shot_smoke(n=5):
    processor = WhisperProcessor.from_pretrained("openai/whisper-base.en")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base.en").to(device)
    model.eval()
    rows = []
    for i in range(n):
        path = splits["test"]["audio_path"][i]
        sentence = splits["test"]["sentence"][i]
        audio_arr, sr = sf.read(path)
        audio_arr = audio_arr.astype(np.float32)
        inputs = processor(audio_arr, return_tensors="pt", sampling_rate=sr)
        input_features = inputs.input_features.to(device)
        with torch.no_grad():
            generated_ids = model.generate(input_features=input_features)
        pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        rows.append({"idx": i, "reference": processor.tokenizer.normalize(sentence), "prediction": processor.tokenizer.normalize(pred)})
    return pd.DataFrame(rows)
df_a7 = whisper_zero_shot_smoke(5)
display(df_a7)
check("A7 Whisper zero-shot smoke", df_a7["prediction"].notna().all(), hard=True)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


,idx,reference,prediction
0,0,average,average
1,1,chalk,chuck
2,2,cliffhanger,cliffhanger
3,3,hoof,huh
4,4,lawyer,lawyer


[PASS] A7 Whisper zero-shot smoke


In [ ]:

def w2v_zero_shot_smoke(n=5):
    processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
    model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h").to(device)
    model.eval()
    rows = []
    for i in range(n):
        path = splits["test"]["audio_path"][i]
        sentence = splits["test"]["sentence"][i]
        audio_arr, sr = sf.read(path)
        audio_arr = audio_arr.astype(np.float32)
        inputs = processor(audio_arr, sampling_rate=sr, return_tensors="pt", padding=True)
        input_values = inputs.input_values.to(device)
        attention_mask = inputs.attention_mask.to(device) if "attention_mask" in inputs else None
        with torch.no_grad():
            logits = model(input_values, attention_mask=attention_mask).logits
        pred_ids = torch.argmax(logits, dim=-1)
        pred = processor.batch_decode(pred_ids)[0]
        rows.append({"idx": i, "reference": normalize_w2v_text(sentence), "prediction": normalize_w2v_text(pred)})
    return pd.DataFrame(rows)
df_a72 = w2v_zero_shot_smoke(5)
display(df_a72)
check("A7.2 W2V zero-shot smoke", df_a72["prediction"].notna().all(), hard=True)

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,idx,reference,prediction
0,0,AVERAGE,AREGE
1,1,CHALK,CACK
2,2,CLIFFHANGER,CLOTH HANGER
3,3,HOOF,A
4,4,LAWYER,LAWYER


[PASS] A7.2 W2V zero-shot smoke


In [ ]:

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h", ctc_zero_infinity=True)
model.config.ctc_zero_infinity = True
model.freeze_feature_encoder()
model = model.to(device)
model.eval()
features = []
for i in range(4):
    path = splits["train"]["audio_path"][i]
    sentence = splits["train"]["sentence"][i]
    audio_arr, sr = sf.read(path)
    audio_arr = audio_arr.astype(np.float32)
    input_values = processor(audio_arr, sampling_rate=sr).input_values[0]
    labels = processor.tokenizer(normalize_w2v_text(sentence)).input_ids
    features.append({"input_values": input_values, "labels": labels, "input_length": len(input_values), "label_length": len(labels)})
batch = DataCollatorCTCWithPadding(processor=processor, padding=True)(features)
batch = {k: v.to(device) for k, v in batch.items()}
with torch.no_grad():
    out = model(**batch)
print("loss =", out.loss.item())
print("loss is finite =", torch.isfinite(out.loss).item())
check("A8 single-batch W2V loss finite", bool(torch.isfinite(out.loss).item()), hard=True)

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


loss = 100.09716033935547
loss is finite = True
[PASS] A8 single-batch W2V loss finite


In [ ]:

def run_whisper_experiment(
    model_name="openai/whisper-base.en",
    do_finetune=False,
    learning_rate=5e-6,
    batch_size=16,
    max_steps=500,
    train_size=None,
    train_age_range=None,
    test_age_range=None,
    augmentation=False,
    output_dir="./whisper_run",
    notes="",
):
    clear_mem()
    splits_local = build_ogi_splits(train_size=train_size, train_age_range=train_age_range, test_age_range=test_age_range, seed=SEED)
    processor = WhisperProcessor.from_pretrained(model_name)
    model = WhisperForConditionalGeneration.from_pretrained(model_name)
    model_input_name = processor.feature_extractor.model_input_names[0]

    def prepare_example(path, sentence, is_train=False):
        audio_arr, sr = sf.read(path)
        audio_arr = audio_arr.astype(np.float32)
        if is_train and augmentation:
            audio_arr = maybe_augment_audio(audio_arr)
        inputs = processor.feature_extractor(audio_arr, sampling_rate=sr)
        labels = processor.tokenizer(processor.tokenizer.normalize(sentence)).input_ids
        return {"input_features": inputs.get(model_input_name)[0], "labels": labels, "input_length": len(audio_arr)}

    if do_finetune:
        train_features = [prepare_example(splits_local["train"]["audio_path"][i], splits_local["train"]["sentence"][i], True) for i in range(len(splits_local["train"]))]
        valid_features = [prepare_example(splits_local["valid"]["audio_path"][i], splits_local["valid"]["sentence"][i], False) for i in range(len(splits_local["valid"]))]
        class SimpleFeatureDataset(torch.utils.data.Dataset):
            def __init__(self, rows): self.rows = rows
            def __len__(self): return len(self.rows)
            def __getitem__(self, idx): return self.rows[idx]
        train_ds = SimpleFeatureDataset(train_features)
        valid_ds = SimpleFeatureDataset(valid_features)
        data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor, decoder_start_token_id=model.config.decoder_start_token_id)

        def compute_metrics(pred):
            pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
            pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
            pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
            label_str = processor.tokenizer.batch_decode(pred.label_ids, skip_special_tokens=True)
            pred_proc, label_proc = [], []
            for p, l in zip(pred_str, label_str):
                if p.strip() and l.strip():
                    pred_proc.append(p); label_proc.append(l)
            return {"wer": wer_metric.compute(predictions=pred_proc, references=label_proc)}

        make_fresh_output_dir(output_dir)
        training_args = Seq2SeqTrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=max(1, batch_size // 2),
            eval_strategy="steps",
            eval_steps=max_steps,
            save_steps=max_steps,
            logging_steps=100,
            max_steps=max_steps,
            learning_rate=learning_rate,
            save_total_limit=2,
            predict_with_generate=True,
            fp16=torch.cuda.is_available(),
            report_to="none",
        )
        trainer = Seq2SeqTrainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=valid_ds,
                                 data_collator=data_collator, compute_metrics=compute_metrics, processing_class=processor)
        trainer.train()
        valid_wer = None
        for log in reversed(trainer.state.log_history):
            if "eval_wer" in log:
                valid_wer = log["eval_wer"]
                break
    else:
        valid_wer = None

    model.to(device); model.eval()
    references, transcriptions = [], []
    for i in range(len(splits_local["test"])):
        path = splits_local["test"]["audio_path"][i]
        sentence = splits_local["test"]["sentence"][i]
        audio_arr, sr = sf.read(path)
        audio_arr = audio_arr.astype(np.float32)
        if len(audio_arr) / sr > 30:
            inputs = processor(audio_arr, return_tensors="pt", truncation=False, padding="longest",
                               return_attention_mask=True, sampling_rate=sr).to(device)
            with torch.no_grad():
                segments = model.generate(**inputs, return_segments=True)["segments"][0]
            prediction = " ".join([processor.decode(seg["tokens"], skip_special_tokens=True) for seg in segments])
        else:
            inputs = processor(audio_arr, return_tensors="pt", sampling_rate=sr)
            with torch.no_grad():
                generated_ids = model.generate(input_features=inputs.input_features.to(device))
            prediction = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        references.append(processor.tokenizer.normalize(sentence))
        transcriptions.append(processor.tokenizer.normalize(prediction))
        if i % 200 == 0:
            print(f"[{model_name}] processed {i} utterances", flush=True)
    test_wer = wer_metric.compute(references=references, predictions=transcriptions)
    add_result("whisper", model_name, "full_ft" if do_finetune else "zero_shot",
               learning_rate if do_finetune else None, batch_size if do_finetune else None, max_steps if do_finetune else None,
               len(splits_local["train"]), str(train_age_range) if train_age_range is not None else "all",
               str(test_age_range) if test_age_range is not None else "all", valid_wer, test_wer,
               "on" if augmentation else "none", notes)
    print("valid_wer =", valid_wer)
    print("test_wer  =", test_wer)
    check(f"{notes} stability", np.isfinite(test_wer) and (test_wer < 1.0), detail=f"test_wer={test_wer}", hard=False)
    save_results_snapshot()
    return {"valid_wer": valid_wer, "test_wer": test_wer}

In [ ]:

def apply_w2v_stability_patch(model):
    model.config.apply_spec_augment = False
    model.config.mask_time_prob = 0.0
    model.config.mask_feature_prob = 0.0
    if hasattr(model.wav2vec2, "masked_spec_embed"):
        model.wav2vec2.masked_spec_embed.requires_grad = False
        with torch.no_grad():
            if not torch.isfinite(model.wav2vec2.masked_spec_embed).all():
                model.wav2vec2.masked_spec_embed.zero_()

def run_w2v_experiment(
    model_name="facebook/wav2vec2-base-960h",
    do_finetune=False,
    freeze_feature_encoder=True,
    learning_rate=1e-5,
    batch_size=8,
    max_steps=1000,
    train_size=None,
    train_age_range=None,
    test_age_range=None,
    augmentation=False,
    output_dir="./w2v_run",
    notes="",
):
    clear_mem()
    splits_local = build_ogi_splits(train_size=train_size, train_age_range=train_age_range, test_age_range=test_age_range, seed=SEED)
    processor = Wav2Vec2Processor.from_pretrained(model_name)
    model = Wav2Vec2ForCTC.from_pretrained(model_name, ctc_zero_infinity=True)
    model.config.ctc_zero_infinity = True
    if freeze_feature_encoder:
        model.freeze_feature_encoder()
    apply_w2v_stability_patch(model)

    def prepare_example(path, sentence, is_train=False):
        audio_arr, sr = sf.read(path)
        audio_arr = audio_arr.astype(np.float32)
        if is_train and augmentation:
            audio_arr = maybe_augment_audio(audio_arr)
        input_values = processor(audio_arr, sampling_rate=sr).input_values[0]
        labels = processor.tokenizer(normalize_w2v_text(sentence)).input_ids
        return {"input_values": input_values, "labels": labels, "input_length": len(input_values), "label_length": len(labels)}

    if do_finetune:
        train_features, valid_features = [], []
        for i in range(len(splits_local["train"])):
            ex = prepare_example(splits_local["train"]["audio_path"][i], splits_local["train"]["sentence"][i], True)
            if valid_ctc_sample_original(ex):
                train_features.append(ex)
        for i in range(len(splits_local["valid"])):
            ex = prepare_example(splits_local["valid"]["audio_path"][i], splits_local["valid"]["sentence"][i], False)
            if valid_ctc_sample_original(ex):
                valid_features.append(ex)
        print("after original CTC filter:", len(train_features), len(valid_features))
        check("W2V train features kept", len(train_features) > 0, hard=True)
        check("W2V valid features kept", len(valid_features) > 0, hard=True)

        class SimpleFeatureDataset(torch.utils.data.Dataset):
            def __init__(self, rows): self.rows = rows
            def __len__(self): return len(self.rows)
            def __getitem__(self, idx): return self.rows[idx]

        train_ds = SimpleFeatureDataset(train_features)
        valid_ds = SimpleFeatureDataset(valid_features)
        data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

        def compute_w2v_metrics(pred):
            pred_ids = np.argmax(pred.predictions, axis=-1)
            pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
            pred_str = processor.batch_decode(pred_ids)
            label_str = processor.tokenizer.batch_decode(pred.label_ids, group_tokens=False)
            pred_proc, label_proc = [], []
            for p, l in zip(pred_str, label_str):
                p_norm = normalize_w2v_text(p); l_norm = normalize_w2v_text(l)
                if p_norm and l_norm:
                    pred_proc.append(p_norm); label_proc.append(l_norm)
            if not label_proc:
                return {"wer": 1.0}
            return {"wer": wer_metric.compute(predictions=pred_proc, references=label_proc)}

        make_fresh_output_dir(output_dir)
        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=max(1, batch_size // 2),
            eval_strategy="steps",
            eval_steps=max_steps,
            save_steps=max_steps,
            logging_steps=100,
            max_steps=max_steps,
            learning_rate=learning_rate,
            warmup_steps=min(10, max_steps // 5),
            max_grad_norm=0.5,
            save_total_limit=2,
            fp16=False,
            report_to="none",
        )
        trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=valid_ds,
                          data_collator=data_collator, processing_class=processor.feature_extractor, compute_metrics=compute_w2v_metrics)
        trainer.train()
        valid_wer = None
        for log in reversed(trainer.state.log_history):
            if "eval_wer" in log:
                valid_wer = log["eval_wer"]
                break
    else:
        valid_wer = None

    model.to(device); model.eval()
    references, transcriptions = [], []
    for i in range(len(splits_local["test"])):
        path = splits_local["test"]["audio_path"][i]
        sentence = splits_local["test"]["sentence"][i]
        audio_arr, sr = sf.read(path)
        audio_arr = audio_arr.astype(np.float32)
        inputs = processor(audio_arr, sampling_rate=sr, return_tensors="pt", padding=True)
        input_values = inputs.input_values.to(device)
        attention_mask = inputs.attention_mask.to(device) if "attention_mask" in inputs else None
        with torch.no_grad():
            logits = model(input_values, attention_mask=attention_mask).logits
        pred_ids = torch.argmax(logits, dim=-1)
        prediction = processor.batch_decode(pred_ids)[0]
        references.append(normalize_w2v_text(sentence))
        transcriptions.append(normalize_w2v_text(prediction))
        if i % 200 == 0:
            print(f"[{model_name}] processed {i} utterances", flush=True)
    test_wer = wer_metric.compute(references=references, predictions=transcriptions)
    add_result("wav2vec2", model_name, "ft_frozen_stable" if do_finetune else "zero_shot",
               learning_rate if do_finetune else None, batch_size if do_finetune else None, max_steps if do_finetune else None,
               len(splits_local["train"]), str(train_age_range) if train_age_range is not None else "all",
               str(test_age_range) if test_age_range is not None else "all", valid_wer, test_wer,
               "on" if augmentation else "none", notes)
    print("valid_wer =", valid_wer)
    print("test_wer  =", test_wer)
    check(f"{notes} stability", np.isfinite(test_wer) and (test_wer < 1.0), detail=f"test_wer={test_wer}", hard=False)
    save_results_snapshot()
    return {"valid_wer": valid_wer, "test_wer": test_wer}

In [ ]:

# Task 1
run_whisper_experiment("openai/whisper-tiny.en", False, notes="task1 whisper tiny zero-shot")
run_whisper_experiment("openai/whisper-tiny.en", True, 5e-6, 16, 500, output_dir="./whisper_tiny_ft", notes="task1 whisper tiny full FT")
run_whisper_experiment("openai/whisper-base.en", False, notes="task1 whisper base zero-shot")
run_whisper_experiment("openai/whisper-base.en", True, 5e-6, 16, 500, output_dir="./whisper_base_ft", notes="task1 whisper base full FT")
display(results_df())

Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/text


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Step,Training Loss,Validation Loss,Wer
500,0.315000,0.485107,0.219311


[openai/whisper-tiny.en] processed 0 utterances
[openai/whisper-tiny.en] processed 200 utterances
[openai/whisper-tiny.en] processed 400 utterances
[openai/whisper-tiny.en] processed 600 utterances
[openai/whisper-tiny.en] processed 800 utterances
[openai/whisper-tiny.en] processed 1000 utterances
[openai/whisper-tiny.en] processed 1200 utterances
[openai/whisper-tiny.en] processed 1400 utterances
[openai/whisper-tiny.en] processed 1600 utterances
[openai/whisper-tiny.en] processed 1800 utterances
valid_wer = 0.2193111029681202
test_wer  = 0.32629744384198295
[PASS] task1 whisper tiny full FT stability :: test_wer=0.32629744384198295
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_2

Step,Training Loss,Validation Loss,Wer
500,0.214600,0.364777,0.160132


[openai/whisper-base.en] processed 0 utterances
[openai/whisper-base.en] processed 200 utterances
[openai/whisper-base.en] processed 400 utterances
[openai/whisper-base.en] processed 600 utterances
[openai/whisper-base.en] processed 800 utterances
[openai/whisper-base.en] processed 1000 utterances
[openai/whisper-base.en] processed 1200 utterances
[openai/whisper-base.en] processed 1400 utterances
[openai/whisper-base.en] processed 1600 utterances
[openai/whisper-base.en] processed 1800 utterances
valid_wer = 0.16013191645291316
test_wer  = 0.2571649883810999
[PASS] task1 whisper base full FT stability :: test_wer=0.2571649883810999
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv


,family,model_name,setting,learning_rate,batch_size,max_steps,train_size,train_age_group,test_age_group,augmentation,valid_wer,test_wer,notes
0,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,500.0,20000,all,all,none,0.160132,0.257165,task1 whisper base full FT
1,whisper,openai/whisper-base.en,zero_shot,NaN,NaN,NaN,20000,all,all,none,NaN,0.285631,task1 whisper base zero-shot
2,whisper,openai/whisper-tiny.en,full_ft,0.000005,16.0,500.0,20000,all,all,none,0.219311,0.326297,task1 whisper tiny full FT
3,whisper,openai/whisper-tiny.en,zero_shot,NaN,NaN,NaN,20000,all,all,none,NaN,0.680480,task1 whisper tiny zero-shot


In [ ]:

# Stabilized full W2V fine-tuning check
stable_w2v_ft_check = run_w2v_experiment(
    do_finetune=True,
    freeze_feature_encoder=True,
    learning_rate=1e-5,
    batch_size=8,
    max_steps=1000,
    train_size=None,
    output_dir="./w2v_finetuned_stable_check",
    notes="stabilized w2v full check"
)
display(results_df())

Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/text


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


after original CTC filter: 20000 2000
[PASS] W2V train features kept
[PASS] W2V valid features kept


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:

# Task 2
run_w2v_experiment(do_finetune=False, notes="task2 wav2vec2 zero-shot")
run_w2v_experiment(do_finetune=True, freeze_feature_encoder=True, learning_rate=1e-5, batch_size=8, max_steps=1000,
                   output_dir="./w2v_base_frozen_ft_stable", notes="task2 wav2vec2 frozen-encoder FT stabilized")
display(results_df())

Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/text


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[facebook/wav2vec2-base-960h] processed 0 utterances
[facebook/wav2vec2-base-960h] processed 200 utterances
[facebook/wav2vec2-base-960h] processed 400 utterances
[facebook/wav2vec2-base-960h] processed 600 utterances
[facebook/wav2vec2-base-960h] processed 800 utterances
[facebook/wav2vec2-base-960h] processed 1000 utterances
[facebook/wav2vec2-base-960h] processed 1200 utterances
[facebook/wav2vec2-base-960h] processed 1400 utterances
[facebook/wav2vec2-base-960h] processed 1600 utterances
[facebook/wav2vec2-base-960h] processed 1800 utterances
valid_wer = None
test_wer  = 0.48210992253780893
[PASS] task2 wav2vec2 zero-shot stability :: test_wer=0.48210992253780893
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


after original CTC filter: 20000 2000
[PASS] W2V train features kept
[PASS] W2V valid features kept


Step,Training Loss,Validation Loss,Wer
1000,95.092800,38.544853,0.292978


[facebook/wav2vec2-base-960h] processed 0 utterances
[facebook/wav2vec2-base-960h] processed 200 utterances
[facebook/wav2vec2-base-960h] processed 400 utterances
[facebook/wav2vec2-base-960h] processed 600 utterances
[facebook/wav2vec2-base-960h] processed 800 utterances
[facebook/wav2vec2-base-960h] processed 1000 utterances
[facebook/wav2vec2-base-960h] processed 1200 utterances
[facebook/wav2vec2-base-960h] processed 1400 utterances
[facebook/wav2vec2-base-960h] processed 1600 utterances
[facebook/wav2vec2-base-960h] processed 1800 utterances
valid_wer = 0.2929782082324455
test_wer  = 0.3423091110291405
[PASS] task2 wav2vec2 frozen-encoder FT stabilized stability :: test_wer=0.3423091110291405
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv


,family,model_name,setting,learning_rate,batch_size,max_steps,train_size,train_age_group,test_age_group,augmentation,valid_wer,test_wer,notes
0,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.292978,0.342309,task2 wav2vec2 frozen-encoder FT stabilized
1,wav2vec2,facebook/wav2vec2-base-960h,zero_shot,NaN,NaN,NaN,20000,all,all,none,NaN,0.482110,task2 wav2vec2 zero-shot
2,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,500.0,20000,all,all,none,0.160132,0.257165,task1 whisper base full FT
3,whisper,openai/whisper-base.en,zero_shot,NaN,NaN,NaN,20000,all,all,none,NaN,0.285631,task1 whisper base zero-shot
4,whisper,openai/whisper-tiny.en,full_ft,0.000005,16.0,500.0,20000,all,all,none,0.219311,0.326297,task1 whisper tiny full FT
5,whisper,openai/whisper-tiny.en,zero_shot,NaN,NaN,NaN,20000,all,all,none,NaN,0.680480,task1 whisper tiny zero-shot


In [ ]:

# Task 3
for lr in [1e-6, 5e-6, 1e-5]:
    run_whisper_experiment("openai/whisper-base.en", True, lr, 16, 500, output_dir=f"./whisper_base_lr_{lr}", notes="task3 whisper lr sweep")
for bs in [8, 16]:
    run_whisper_experiment("openai/whisper-base.en", True, 5e-6, bs, 500, output_dir=f"./whisper_base_bs_{bs}", notes="task3 whisper batch sweep")
for steps in [500, 1000, 2000]:
    run_whisper_experiment("openai/whisper-base.en", True, 5e-6, 16, steps, output_dir=f"./whisper_base_steps_{steps}", notes="task3 whisper steps sweep")

for lr in [3e-6, 1e-5, 3e-5]:
    run_w2v_experiment(do_finetune=True, freeze_feature_encoder=True, learning_rate=lr, batch_size=8, max_steps=1000,
                       output_dir=f"./w2v_base_lr_{lr}_stable", notes="task3 w2v lr sweep stabilized")
for bs in [8, 16]:
    run_w2v_experiment(do_finetune=True, freeze_feature_encoder=True, learning_rate=1e-5, batch_size=bs, max_steps=1000,
                       output_dir=f"./w2v_base_bs_{bs}_stable", notes="task3 w2v batch sweep stabilized")
for steps in [500, 1000, 2000]:
    run_w2v_experiment(do_finetune=True, freeze_feature_encoder=True, learning_rate=1e-5, batch_size=8, max_steps=steps,
                       output_dir=f"./w2v_base_steps_{steps}_stable", notes="task3 w2v steps sweep stabilized")
display(results_df())

Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/text


Step,Training Loss,Validation Loss,Wer
500,0.493300,0.568823,0.244412


[openai/whisper-base.en] processed 0 utterances
[openai/whisper-base.en] processed 200 utterances
[openai/whisper-base.en] processed 400 utterances
[openai/whisper-base.en] processed 600 utterances
[openai/whisper-base.en] processed 800 utterances
[openai/whisper-base.en] processed 1000 utterances
[openai/whisper-base.en] processed 1200 utterances
[openai/whisper-base.en] processed 1400 utterances
[openai/whisper-base.en] processed 1600 utterances
[openai/whisper-base.en] processed 1800 utterances
valid_wer = 0.24441187248076218
test_wer  = 0.26626646010844307
[PASS] task3 whisper lr sweep stability :: test_wer=0.26626646010844307
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B

Step,Training Loss,Validation Loss,Wer
500,0.214600,0.364750,0.160132


[openai/whisper-base.en] processed 0 utterances
[openai/whisper-base.en] processed 200 utterances
[openai/whisper-base.en] processed 400 utterances
[openai/whisper-base.en] processed 600 utterances
[openai/whisper-base.en] processed 800 utterances
[openai/whisper-base.en] processed 1000 utterances
[openai/whisper-base.en] processed 1200 utterances
[openai/whisper-base.en] processed 1400 utterances
[openai/whisper-base.en] processed 1600 utterances
[openai/whisper-base.en] processed 1800 utterances
valid_wer = 0.16013191645291316
test_wer  = 0.2569713400464756
[PASS] task3 whisper lr sweep stability :: test_wer=0.2569713400464756
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_M

...

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


after original CTC filter: 20000 2000
[PASS] W2V train features kept
[PASS] W2V valid features kept


Step,Training Loss,Validation Loss,Wer
2000,82.536000,34.248257,0.245933


[facebook/wav2vec2-base-960h] processed 0 utterances
[facebook/wav2vec2-base-960h] processed 200 utterances
[facebook/wav2vec2-base-960h] processed 400 utterances
[facebook/wav2vec2-base-960h] processed 600 utterances
[facebook/wav2vec2-base-960h] processed 800 utterances
[facebook/wav2vec2-base-960h] processed 1000 utterances
[facebook/wav2vec2-base-960h] processed 1200 utterances
[facebook/wav2vec2-base-960h] processed 1400 utterances
[facebook/wav2vec2-base-960h] processed 1600 utterances
[facebook/wav2vec2-base-960h] processed 1800 utterances
valid_wer = 0.24593284873658705
test_wer  = 0.32054592401327925
[PASS] task3 w2v steps sweep stabilized stability :: test_wer=0.32054592401327925
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv


,family,model_name,setting,learning_rate,batch_size,max_steps,train_size,train_age_group,test_age_group,augmentation,valid_wer,test_wer,notes
0,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,20000,all,all,none,0.235253,0.311324,task3 w2v lr sweep stabilized
1,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,2000.0,20000,all,all,none,0.245933,0.320546,task3 w2v steps sweep stabilized
2,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,16.0,1000.0,20000,all,all,none,0.267947,0.330505,task3 w2v batch sweep stabilized
3,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.288751,0.338620,task3 w2v lr sweep stabilized
4,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.288704,0.339358,task3 w2v batch sweep stabilized
5,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.292978,0.342309,task2 wav2vec2 frozen-encoder FT stabilized
6,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.299706,0.345260,task3 w2v steps sweep stabilized
7,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,500.0,20000,all,all,none,0.321720,0.364257,task3 w2v steps sweep stabilized
8,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000003,8.0,1000.0,20000,all,all,none,0.341928,0.377167,task3 w2v lr sweep stabilized
9,wav2vec2,facebook/wav2vec2-base-960h,zero_shot,NaN,NaN,NaN,20000,all,all,none,NaN,0.482110,task2 wav2vec2 zero-shot


In [ ]:

df = results_df().copy()
task3 = df[df["notes"].str.contains("task3", case=False, na=False)].copy()
task3 = task3[task3["valid_wer"].notna() & np.isfinite(task3["valid_wer"])].copy()
task3 = task3[task3["valid_wer"] < 1.0].copy()
display(task3[["family", "model_name", "learning_rate", "batch_size", "max_steps", "valid_wer", "test_wer", "notes"]].sort_values(["family", "valid_wer", "test_wer"]))

best_whisper = task3[task3["family"] == "whisper"].sort_values(["valid_wer", "test_wer"], ascending=[True, True]).iloc[0]
best_w2v = task3[task3["family"] == "wav2vec2"].sort_values(["valid_wer", "test_wer"], ascending=[True, True]).iloc[0]

BEST_WHISPER_LR = float(best_whisper["learning_rate"]); BEST_WHISPER_BS = int(best_whisper["batch_size"]); BEST_WHISPER_STEPS = int(best_whisper["max_steps"])
BEST_W2V_LR = float(best_w2v["learning_rate"]); BEST_W2V_BS = int(best_w2v["batch_size"]); BEST_W2V_STEPS = int(best_w2v["max_steps"])

print("Selected best Task 3 configs:")
print(f"Whisper  -> lr={BEST_WHISPER_LR}, batch={BEST_WHISPER_BS}, steps={BEST_WHISPER_STEPS}, valid_wer={best_whisper['valid_wer']:.6f}, test_wer={best_whisper['test_wer']:.6f}")
print(f"Wav2Vec2 -> lr={BEST_W2V_LR}, batch={BEST_W2V_BS}, steps={BEST_W2V_STEPS}, valid_wer={best_w2v['valid_wer']:.6f}, test_wer={best_w2v['test_wer']:.6f}")

,family,model_name,learning_rate,batch_size,max_steps,valid_wer,test_wer,notes
0,wav2vec2,facebook/wav2vec2-base-960h,0.000030,8.0,1000.0,0.235253,0.311324,task3 w2v lr sweep stabilized
1,wav2vec2,facebook/wav2vec2-base-960h,0.000010,8.0,2000.0,0.245933,0.320546,task3 w2v steps sweep stabilized
2,wav2vec2,facebook/wav2vec2-base-960h,0.000010,16.0,1000.0,0.267947,0.330505,task3 w2v batch sweep stabilized
4,wav2vec2,facebook/wav2vec2-base-960h,0.000010,8.0,1000.0,0.288704,0.339358,task3 w2v batch sweep stabilized
3,wav2vec2,facebook/wav2vec2-base-960h,0.000010,8.0,1000.0,0.288751,0.338620,task3 w2v lr sweep stabilized
6,wav2vec2,facebook/wav2vec2-base-960h,0.000010,8.0,1000.0,0.299706,0.345260,task3 w2v steps sweep stabilized
7,wav2vec2,facebook/wav2vec2-base-960h,0.000010,8.0,500.0,0.321720,0.364257,task3 w2v steps sweep stabilized
8,wav2vec2,facebook/wav2vec2-base-960h,0.000003,8.0,1000.0,0.341928,0.377167,task3 w2v lr sweep stabilized
19,whisper,openai/whisper-base.en,0.000005,16.0,2000.0,0.147123,0.320101,task3 whisper steps sweep
17,whisper,openai/whisper-base.en,0.000005,16.0,1000.0,0.148223,0.275949,task3 whisper steps sweep


Selected best Task 3 configs:
Whisper  -> lr=5e-06, batch=16, steps=2000, valid_wer=0.147123, test_wer=0.320101
Wav2Vec2 -> lr=3e-05, batch=8, steps=1000, valid_wer=0.235253, test_wer=0.311324


In [ ]:

# Task 4
for n in [20000, 10000, 5000]:
    run_whisper_experiment("openai/whisper-base.en", True, BEST_WHISPER_LR, BEST_WHISPER_BS, BEST_WHISPER_STEPS,
                           train_size=n, output_dir=f"./whisper_base_size_{n}", notes="task4 whisper size sweep")
for n in [20000, 10000, 5000]:
    run_w2v_experiment(do_finetune=True, freeze_feature_encoder=True, learning_rate=BEST_W2V_LR, batch_size=BEST_W2V_BS, max_steps=BEST_W2V_STEPS,
                       train_size=n, output_dir=f"./w2v_base_size_{n}_stable", notes="task4 w2v size sweep stabilized")
display(results_df())

Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/text


Step,Training Loss,Validation Loss,Wer
2000,0.065500,0.309682,0.152070


[openai/whisper-base.en] processed 0 utterances
[openai/whisper-base.en] processed 200 utterances
[openai/whisper-base.en] processed 400 utterances
[openai/whisper-base.en] processed 600 utterances
[openai/whisper-base.en] processed 800 utterances
[openai/whisper-base.en] processed 1000 utterances
[openai/whisper-base.en] processed 1200 utterances
[openai/whisper-base.en] processed 1400 utterances
[openai/whisper-base.en] processed 1600 utterances
[openai/whisper-base.en] processed 1800 utterances
valid_wer = 0.15207035544155367
test_wer  = 0.31661502711076683
[PASS] task4 whisper size sweep stability :: test_wer=0.31661502711076683
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_21

Step,Training Loss,Validation Loss,Wer
2000,0.045000,0.311809,0.154269


[openai/whisper-base.en] processed 0 utterances
[openai/whisper-base.en] processed 200 utterances
[openai/whisper-base.en] processed 400 utterances
[openai/whisper-base.en] processed 600 utterances
[openai/whisper-base.en] processed 800 utterances
[openai/whisper-base.en] processed 1000 utterances
[openai/whisper-base.en] processed 1200 utterances
[openai/whisper-base.en] processed 1400 utterances
[openai/whisper-base.en] processed 1600 utterances
[openai/whisper-base.en] processed 1800 utterances
valid_wer = 0.15426896299010626
test_wer  = 0.31835786212238576
[PASS] task4 whisper size sweep stability :: test_wer=0.31835786212238576
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_21

...

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


after original CTC filter: 5000 2000
[PASS] W2V train features kept
[PASS] W2V valid features kept


Step,Training Loss,Validation Loss,Wer
1000,67.537900,32.949665,0.232703


[facebook/wav2vec2-base-960h] processed 0 utterances
[facebook/wav2vec2-base-960h] processed 200 utterances
[facebook/wav2vec2-base-960h] processed 400 utterances
[facebook/wav2vec2-base-960h] processed 600 utterances
[facebook/wav2vec2-base-960h] processed 800 utterances
[facebook/wav2vec2-base-960h] processed 1000 utterances
[facebook/wav2vec2-base-960h] processed 1200 utterances
[facebook/wav2vec2-base-960h] processed 1400 utterances
[facebook/wav2vec2-base-960h] processed 1600 utterances
[facebook/wav2vec2-base-960h] processed 1800 utterances
valid_wer = 0.23270331194728627
test_wer  = 0.3199926226484692
[PASS] task4 w2v size sweep stabilized stability :: test_wer=0.3199926226484692
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv


,family,model_name,setting,learning_rate,batch_size,max_steps,train_size,train_age_group,test_age_group,augmentation,valid_wer,test_wer,notes
0,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,20000,all,all,none,0.235253,0.311324,task3 w2v lr sweep stabilized
1,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,20000,all,all,none,0.229732,0.316488,task4 w2v size sweep stabilized
2,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,5000,all,all,none,0.232703,0.319993,task4 w2v size sweep stabilized
3,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,2000.0,20000,all,all,none,0.245933,0.320546,task3 w2v steps sweep stabilized
4,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,10000,all,all,none,0.230397,0.321284,task4 w2v size sweep stabilized
5,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,16.0,1000.0,20000,all,all,none,0.267947,0.330505,task3 w2v batch sweep stabilized
6,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.288751,0.338620,task3 w2v lr sweep stabilized
7,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.288704,0.339358,task3 w2v batch sweep stabilized
8,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.292978,0.342309,task2 wav2vec2 frozen-encoder FT stabilized
9,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.299706,0.345260,task3 w2v steps sweep stabilized


In [ ]:
# Run checkpoint to save results in case
df = results_df()

checkpoint_csv_path = "/content/drive/MyDrive/ogi_kids_asr_results_checkpoint_after_task4.csv"
df.to_csv(checkpoint_csv_path, index=False)

print("Checkpoint saved to:", checkpoint_csv_path)
display(df.tail(20))

Checkpoint saved to: /content/drive/MyDrive/ogi_kids_asr_results_checkpoint_after_task4.csv


,family,model_name,setting,learning_rate,batch_size,max_steps,train_size,train_age_group,test_age_group,augmentation,valid_wer,test_wer,notes
8,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.292978,0.342309,task2 wav2vec2 frozen-encoder FT stabilized
9,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.299706,0.345260,task3 w2v steps sweep stabilized
10,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,500.0,20000,all,all,none,0.321720,0.364257,task3 w2v steps sweep stabilized
11,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000003,8.0,1000.0,20000,all,all,none,0.341928,0.377167,task3 w2v lr sweep stabilized
12,wav2vec2,facebook/wav2vec2-base-960h,zero_shot,NaN,NaN,NaN,20000,all,all,none,NaN,0.482110,task2 wav2vec2 zero-shot
13,whisper,openai/whisper-base.en,full_ft,0.000005,8.0,500.0,20000,all,all,none,0.171675,0.253873,task3 whisper batch sweep
14,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,500.0,20000,all,all,none,0.160132,0.256971,task3 whisper lr sweep
15,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,500.0,20000,all,all,none,0.159949,0.257165,task3 whisper batch sweep
16,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,500.0,20000,all,all,none,0.159949,0.257165,task3 whisper steps sweep
17,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,500.0,20000,all,all,none,0.160132,0.257165,task1 whisper base full FT


In [ ]:
from datetime import datetime

In [ ]:
# Task 5 — Whisper
age_groups = {"4-6": (4, 6), "7-9": (7, 9), "10-14": (10, 14), "all": None}

for train_name, train_range in age_groups.items():
    for test_name, test_range in age_groups.items():
        run_whisper_experiment(
            "openai/whisper-base.en",
            True,
            BEST_WHISPER_LR,
            BEST_WHISPER_BS,
            BEST_WHISPER_STEPS,
            train_age_range=train_range,
            test_age_range=test_range,
            output_dir=f"./whisper_age_{train_name}_to_{test_name}",
            notes=f"task5 whisper train={train_name} test={test_name}"
        )

display(results_df())

Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/text


Step,Training Loss,Validation Loss,Wer
2000,0.048700,0.576689,0.306999


[openai/whisper-base.en] processed 0 utterances
[openai/whisper-base.en] processed 200 utterances
[openai/whisper-base.en] processed 400 utterances
valid_wer = 0.306999306999307
test_wer  = 0.5855614973262032
[PASS] task5 whisper train=4-6 test=4-6 stability :: test_wer=0.5855614973262032
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/text


Step,Training Loss,Validation Loss,Wer
2000,0.048600,0.577020,0.306999


[openai/whisper-base.en] processed 0 utterances
[openai/whisper-base.en] processed 200 utterances
[openai/whisper-base.en] processed 400 utterances
valid_wer = 0.306999306999307
test_wer  = 0.3736343772760379
[PASS] task5 whisper train=4-6 test=7-9 stability :: test_wer=0.3736343772760379
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/text


...

Step,Training Loss,Validation Loss,Wer
2000,0.050400,0.309270,0.147673


[openai/whisper-base.en] processed 0 utterances
[openai/whisper-base.en] processed 200 utterances
[openai/whisper-base.en] processed 400 utterances
[openai/whisper-base.en] processed 600 utterances
[openai/whisper-base.en] processed 800 utterances
valid_wer = 0.1476731403444485
test_wer  = 0.21525054466230936
[PASS] task5 whisper train=all test=10-14 stability :: test_wer=0.21525054466230936
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/text


Step,Training Loss,Validation Loss,Wer
2000,0.050300,0.309348,0.147673


[openai/whisper-base.en] processed 0 utterances
[openai/whisper-base.en] processed 200 utterances
[openai/whisper-base.en] processed 400 utterances
[openai/whisper-base.en] processed 600 utterances
[openai/whisper-base.en] processed 800 utterances
[openai/whisper-base.en] processed 1000 utterances
[openai/whisper-base.en] processed 1200 utterances
[openai/whisper-base.en] processed 1400 utterances
[openai/whisper-base.en] processed 1600 utterances
[openai/whisper-base.en] processed 1800 utterances
valid_wer = 0.1476731403444485
test_wer  = 0.3206816421378776
[PASS] task5 whisper train=all test=all stability :: test_wer=0.3206816421378776
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv


,family,model_name,setting,learning_rate,batch_size,max_steps,train_size,train_age_group,test_age_group,augmentation,valid_wer,test_wer,notes
0,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,20000,all,all,none,0.235253,0.311324,task3 w2v lr sweep stabilized
1,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,20000,all,all,none,0.229732,0.316488,task4 w2v size sweep stabilized
2,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,5000,all,all,none,0.232703,0.319993,task4 w2v size sweep stabilized
3,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,2000.0,20000,all,all,none,0.245933,0.320546,task3 w2v steps sweep stabilized
4,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,10000,all,all,none,0.230397,0.321284,task4 w2v size sweep stabilized
5,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,16.0,1000.0,20000,all,all,none,0.267947,0.330505,task3 w2v batch sweep stabilized
6,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.288751,0.338620,task3 w2v lr sweep stabilized
7,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.288704,0.339358,task3 w2v batch sweep stabilized
8,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.292978,0.342309,task2 wav2vec2 frozen-encoder FT stabilized
9,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000010,8.0,1000.0,20000,all,all,none,0.299706,0.345260,task3 w2v steps sweep stabilized


In [ ]:
# Save checkpoint after Task 5 Whisper
df = results_df()
checkpoint_csv_path = "/content/drive/MyDrive/ogi_kids_asr_results_checkpoint_after_task5_whisper.csv"
df.to_csv(checkpoint_csv_path, index=False)
print("Checkpoint saved to:", checkpoint_csv_path)

Checkpoint saved to: /content/drive/MyDrive/ogi_kids_asr_results_checkpoint_after_task5_whisper.csv


In [ ]:
# Task 5 — Wav2Vec2
age_groups = {"4-6": (4, 6), "7-9": (7, 9), "10-14": (10, 14), "all": None}

for train_name, train_range in age_groups.items():
    for test_name, test_range in age_groups.items():
        run_w2v_experiment(
            do_finetune=True,
            freeze_feature_encoder=True,
            learning_rate=BEST_W2V_LR,
            batch_size=BEST_W2V_BS,
            max_steps=BEST_W2V_STEPS,
            train_age_range=train_range,
            test_age_range=test_range,
            output_dir=f"./w2v_age_{train_name}_to_{test_name}_stable",
            notes=f"task5 w2v train={train_name} test={test_name} stabilized"
        )

display(results_df())

Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/text


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


after original CTC filter: 5402 544
[PASS] W2V train features kept
[PASS] W2V valid features kept


Step,Training Loss,Validation Loss,Wer
1000,145.261300,68.941528,0.466488


[facebook/wav2vec2-base-960h] processed 0 utterances
[facebook/wav2vec2-base-960h] processed 200 utterances
[facebook/wav2vec2-base-960h] processed 400 utterances
valid_wer = 0.46648793565683644
test_wer  = 0.56
[PASS] task5 w2v train=4-6 test=4-6 stabilized stability :: test_wer=0.56
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/text


...

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


after original CTC filter: 20000 2000
[PASS] W2V train features kept
[PASS] W2V valid features kept


Step,Training Loss,Validation Loss,Wer
1000,72.648900,31.859629,0.230716


[facebook/wav2vec2-base-960h] processed 0 utterances
[facebook/wav2vec2-base-960h] processed 200 utterances
[facebook/wav2vec2-base-960h] processed 400 utterances
[facebook/wav2vec2-base-960h] processed 600 utterances
[facebook/wav2vec2-base-960h] processed 800 utterances
[facebook/wav2vec2-base-960h] processed 1000 utterances
[facebook/wav2vec2-base-960h] processed 1200 utterances
[facebook/wav2vec2-base-960h] processed 1400 utterances
[facebook/wav2vec2-base-960h] processed 1600 utterances
[facebook/wav2vec2-base-960h] processed 1800 utterances
valid_wer = 0.23071601521964719
test_wer  = 0.3133530062707488
[PASS] task5 w2v train=all test=all stabilized stability :: test_wer=0.3133530062707488
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv


,family,model_name,setting,learning_rate,batch_size,max_steps,train_size,train_age_group,test_age_group,augmentation,valid_wer,test_wer,notes
0,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,5475,"(7, 9)","(10, 14)",none,0.177309,0.185232,task5 w2v train=7-9 test=10-14 stabilized
1,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,9123,"(10, 14)","(10, 14)",none,0.116835,0.186900,task5 w2v train=10-14 test=10-14 stabilized
2,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,20000,all,"(10, 14)",none,0.234856,0.187317,task5 w2v train=all test=10-14 stabilized
3,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,5402,"(4, 6)","(10, 14)",none,0.457661,0.208177,task5 w2v train=4-6 test=10-14 stabilized
4,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,5475,"(7, 9)","(7, 9)",none,0.169460,0.261695,task5 w2v train=7-9 test=7-9 stabilized
5,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,5402,"(4, 6)","(7, 9)",none,0.472801,0.284068,task5 w2v train=4-6 test=7-9 stabilized
6,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,20000,all,"(7, 9)",none,0.234888,0.285424,task5 w2v train=all test=7-9 stabilized
7,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,20000,all,all,none,0.235253,0.311324,task3 w2v lr sweep stabilized
8,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,9123,"(10, 14)","(7, 9)",none,0.120941,0.312542,task5 w2v train=10-14 test=7-9 stabilized
9,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,20000,all,all,none,0.230716,0.313353,task5 w2v train=all test=all stabilized


In [ ]:
# Save checkpoint after full Task 5
df = results_df()
checkpoint_csv_path = "/content/drive/MyDrive/ogi_kids_asr_results_checkpoint_after_task5_full.csv"
df.to_csv(checkpoint_csv_path, index=False)
print("Checkpoint saved to:", checkpoint_csv_path)

Checkpoint saved to: /content/drive/MyDrive/ogi_kids_asr_results_checkpoint_after_task5_full.csv


In [ ]:
df = results_df()

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_csv_path = f"/content/drive/MyDrive/ogi_kids_asr_results_checkpoint_{ts}.csv"
df.to_csv(checkpoint_csv_path, index=False)

print("Checkpoint saved to:", checkpoint_csv_path)
display(df.tail(20))

Checkpoint saved to: /content/drive/MyDrive/ogi_kids_asr_results_checkpoint_20260423_081730.csv


,family,model_name,setting,learning_rate,batch_size,max_steps,train_size,train_age_group,test_age_group,augmentation,valid_wer,test_wer,notes
40,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,9123,"(10, 14)",all,none,0.072821,0.275368,task5 whisper train=10-14 test=all
41,whisper,openai/whisper-base.en,full_ft,0.000010,16.0,500.0,20000,all,all,none,0.155185,0.275562,task3 whisper lr sweep
42,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,1000.0,20000,all,all,none,0.148223,0.275949,task3 whisper steps sweep
43,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,20000,all,"(7, 9)",none,0.147123,0.276038,task5 whisper train=all test=7-9
44,whisper,openai/whisper-base.en,zero_shot,NaN,NaN,NaN,20000,all,all,none,NaN,0.285631,task1 whisper base zero-shot
45,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5475,"(7, 9)",all,none,0.091212,0.288149,task5 whisper train=7-9 test=all
46,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5402,"(4, 6)","(10, 14)",none,0.304227,0.290196,task5 whisper train=4-6 test=10-14
47,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5000,all,all,none,0.147856,0.305383,task4 whisper size sweep
48,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,20000,all,all,none,0.152070,0.316615,task4 whisper size sweep
49,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,10000,all,all,none,0.154269,0.318358,task4 whisper size sweep


In [ ]:
df = results_df()
whisper_age = df[(df["family"] == "whisper") & (df["notes"].str.contains("task5"))].pivot_table(index="train_age_group", columns="test_age_group", values="test_wer", aggfunc="min")
w2v_age = df[(df["family"] == "wav2vec2") & (df["notes"].str.contains("task5"))].pivot_table(index="train_age_group", columns="test_age_group", values="test_wer", aggfunc="min")
print("Whisper age-transfer WER")
display(whisper_age)
print("Wav2Vec2 age-transfer WER")
display(w2v_age)

Whisper age-transfer WER


test_age_group,"(10, 14)","(4, 6)","(7, 9)",all
train_age_group,,,,
"(10, 14)",0.178214,0.459893,0.239621,0.275368
"(4, 6)",0.290196,0.585561,0.373634,0.398916
"(7, 9)",0.189542,0.467246,0.254916,0.288149
all,0.215251,0.518048,0.276038,0.320682


Wav2Vec2 age-transfer WER


test_age_group,"(10, 14)","(4, 6)","(7, 9)",all
train_age_group,,,,
"(10, 14)",0.186900,0.594194,0.312542,0.336038
"(4, 6)",0.208177,0.560000,0.284068,0.329768
"(7, 9)",0.185232,0.538710,0.261695,0.316120
all,0.187317,0.551613,0.285424,0.313353


In [ ]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_csv_path = f"/content/drive/MyDrive/ogi_kids_asr_results_checkpoint_{ts}.csv"
df.to_csv(checkpoint_csv_path, index=False)

print("Checkpoint saved to:", checkpoint_csv_path)
display(df.tail(20))

Checkpoint saved to: /content/drive/MyDrive/ogi_kids_asr_results_checkpoint_20260423_081730.csv


,family,model_name,setting,learning_rate,batch_size,max_steps,train_size,train_age_group,test_age_group,augmentation,valid_wer,test_wer,notes
40,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,9123,"(10, 14)",all,none,0.072821,0.275368,task5 whisper train=10-14 test=all
41,whisper,openai/whisper-base.en,full_ft,0.000010,16.0,500.0,20000,all,all,none,0.155185,0.275562,task3 whisper lr sweep
42,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,1000.0,20000,all,all,none,0.148223,0.275949,task3 whisper steps sweep
43,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,20000,all,"(7, 9)",none,0.147123,0.276038,task5 whisper train=all test=7-9
44,whisper,openai/whisper-base.en,zero_shot,NaN,NaN,NaN,20000,all,all,none,NaN,0.285631,task1 whisper base zero-shot
45,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5475,"(7, 9)",all,none,0.091212,0.288149,task5 whisper train=7-9 test=all
46,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5402,"(4, 6)","(10, 14)",none,0.304227,0.290196,task5 whisper train=4-6 test=10-14
47,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5000,all,all,none,0.147856,0.305383,task4 whisper size sweep
48,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,20000,all,all,none,0.152070,0.316615,task4 whisper size sweep
49,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,10000,all,all,none,0.154269,0.318358,task4 whisper size sweep


In [ ]:

# Task 6
run_whisper_experiment("openai/whisper-base.en", True, BEST_WHISPER_LR, BEST_WHISPER_BS, BEST_WHISPER_STEPS,
                       augmentation=False, output_dir="./whisper_base_noaug", notes="task6 whisper no augmentation")
run_whisper_experiment("openai/whisper-base.en", True, BEST_WHISPER_LR, BEST_WHISPER_BS, BEST_WHISPER_STEPS,
                       augmentation=True, output_dir="./whisper_base_aug", notes="task6 whisper with augmentation")
run_w2v_experiment(do_finetune=True, freeze_feature_encoder=True, learning_rate=BEST_W2V_LR, batch_size=BEST_W2V_BS, max_steps=BEST_W2V_STEPS,
                   augmentation=False, output_dir="./w2v_base_noaug_stable", notes="task6 w2v no augmentation stabilized")
run_w2v_experiment(do_finetune=True, freeze_feature_encoder=True, learning_rate=BEST_W2V_LR, batch_size=BEST_W2V_BS, max_steps=BEST_W2V_STEPS,
                   augmentation=True, output_dir="./w2v_base_aug_stable", notes="task6 w2v with augmentation stabilized")
display(results_df())

Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/wav.scp
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/test/text


Step,Training Loss,Validation Loss,Wer
2000,0.050400,0.309297,0.146940


[openai/whisper-base.en] processed 0 utterances
[openai/whisper-base.en] processed 200 utterances
[openai/whisper-base.en] processed 400 utterances
[openai/whisper-base.en] processed 600 utterances
[openai/whisper-base.en] processed 800 utterances
[openai/whisper-base.en] processed 1000 utterances
[openai/whisper-base.en] processed 1200 utterances
[openai/whisper-base.en] processed 1400 utterances
[openai/whisper-base.en] processed 1600 utterances
[openai/whisper-base.en] processed 1800 utterances
valid_wer = 0.14694027116159766
test_wer  = 0.3212625871417506
[PASS] task6 whisper no augmentation stability :: test_wer=0.3212625871417506
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_ECE

Step,Training Loss,Validation Loss,Wer
2000,0.085000,0.327377,0.168193


[openai/whisper-base.en] processed 0 utterances
[openai/whisper-base.en] processed 200 utterances
[openai/whisper-base.en] processed 400 utterances
[openai/whisper-base.en] processed 600 utterances
[openai/whisper-base.en] processed 800 utterances
[openai/whisper-base.en] processed 1000 utterances
[openai/whisper-base.en] processed 1200 utterances
[openai/whisper-base.en] processed 1400 utterances
[openai/whisper-base.en] processed 1600 utterances
[openai/whisper-base.en] processed 1800 utterances
valid_wer = 0.16819347746427263
test_wer  = 0.3687064291247095
[PASS] task6 whisper with augmentation stability :: test_wer=0.3687064291247095
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/wav.scp
Reading 20000 lines from /content/S26_ECE_214B_Mini_Project_1/data/train/text
Reading 2000 lines from /content/S26_ECE_214B_Mini_Project_1/data/dev/wav.scp
Reading 2000 lines from /content/S26_E

...

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


after original CTC filter: 20000 2000
[PASS] W2V train features kept
[PASS] W2V valid features kept


Step,Training Loss,Validation Loss,Wer
1000,117.725700,34.685665,0.258287


[facebook/wav2vec2-base-960h] processed 0 utterances
[facebook/wav2vec2-base-960h] processed 200 utterances
[facebook/wav2vec2-base-960h] processed 400 utterances
[facebook/wav2vec2-base-960h] processed 600 utterances
[facebook/wav2vec2-base-960h] processed 800 utterances
[facebook/wav2vec2-base-960h] processed 1000 utterances
[facebook/wav2vec2-base-960h] processed 1200 utterances
[facebook/wav2vec2-base-960h] processed 1400 utterances
[facebook/wav2vec2-base-960h] processed 1600 utterances
[facebook/wav2vec2-base-960h] processed 1800 utterances
valid_wer = 0.25828729281767954
test_wer  = 0.3247879011434895
[PASS] task6 w2v with augmentation stabilized stability :: test_wer=0.3247879011434895
Autosaved results to: /content/drive/MyDrive/ogi_kids_asr_results_autosave.csv


,family,model_name,setting,learning_rate,batch_size,max_steps,train_size,train_age_group,test_age_group,augmentation,valid_wer,test_wer,notes
0,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,5475,"(7, 9)","(10, 14)",none,0.177309,0.185232,task5 w2v train=7-9 test=10-14 stabilized
1,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,9123,"(10, 14)","(10, 14)",none,0.116835,0.186900,task5 w2v train=10-14 test=10-14 stabilized
2,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,20000,all,"(10, 14)",none,0.234856,0.187317,task5 w2v train=all test=10-14 stabilized
3,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,5402,"(4, 6)","(10, 14)",none,0.457661,0.208177,task5 w2v train=4-6 test=10-14 stabilized
4,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,5475,"(7, 9)","(7, 9)",none,0.169460,0.261695,task5 w2v train=7-9 test=7-9 stabilized
...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,9123,"(10, 14)","(4, 6)",none,0.072821,0.459893,task5 whisper train=10-14 test=4-6
60,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5475,"(7, 9)","(4, 6)",none,0.091212,0.467246,task5 whisper train=7-9 test=4-6
61,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,20000,all,"(4, 6)",none,0.147123,0.518048,task5 whisper train=all test=4-6
62,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5402,"(4, 6)","(4, 6)",none,0.306999,0.585561,task5 whisper train=4-6 test=4-6


In [ ]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_csv_path = f"/content/drive/MyDrive/ogi_kids_asr_results_checkpoint_{ts}.csv"
df.to_csv(checkpoint_csv_path, index=False)

print("Checkpoint saved to:", checkpoint_csv_path)
display(df.tail(20))

Checkpoint saved to: /content/drive/MyDrive/ogi_kids_asr_results_checkpoint_20260423_090808.csv


,family,model_name,setting,learning_rate,batch_size,max_steps,train_size,train_age_group,test_age_group,augmentation,valid_wer,test_wer,notes
40,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,9123,"(10, 14)",all,none,0.072821,0.275368,task5 whisper train=10-14 test=all
41,whisper,openai/whisper-base.en,full_ft,0.000010,16.0,500.0,20000,all,all,none,0.155185,0.275562,task3 whisper lr sweep
42,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,1000.0,20000,all,all,none,0.148223,0.275949,task3 whisper steps sweep
43,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,20000,all,"(7, 9)",none,0.147123,0.276038,task5 whisper train=all test=7-9
44,whisper,openai/whisper-base.en,zero_shot,NaN,NaN,NaN,20000,all,all,none,NaN,0.285631,task1 whisper base zero-shot
45,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5475,"(7, 9)",all,none,0.091212,0.288149,task5 whisper train=7-9 test=all
46,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5402,"(4, 6)","(10, 14)",none,0.304227,0.290196,task5 whisper train=4-6 test=10-14
47,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5000,all,all,none,0.147856,0.305383,task4 whisper size sweep
48,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,20000,all,all,none,0.152070,0.316615,task4 whisper size sweep
49,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,10000,all,all,none,0.154269,0.318358,task4 whisper size sweep


In [ ]:

df = results_df()
final_csv_path = "/content/drive/MyDrive/ogi_kids_asr_results_final.csv"
df.to_csv(final_csv_path, index=False)
print("Saved:", final_csv_path)
display(df)

Saved: /content/drive/MyDrive/ogi_kids_asr_results_final.csv


,family,model_name,setting,learning_rate,batch_size,max_steps,train_size,train_age_group,test_age_group,augmentation,valid_wer,test_wer,notes
0,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,5475,"(7, 9)","(10, 14)",none,0.177309,0.185232,task5 w2v train=7-9 test=10-14 stabilized
1,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,9123,"(10, 14)","(10, 14)",none,0.116835,0.186900,task5 w2v train=10-14 test=10-14 stabilized
2,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,20000,all,"(10, 14)",none,0.234856,0.187317,task5 w2v train=all test=10-14 stabilized
3,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,5402,"(4, 6)","(10, 14)",none,0.457661,0.208177,task5 w2v train=4-6 test=10-14 stabilized
4,wav2vec2,facebook/wav2vec2-base-960h,ft_frozen_stable,0.000030,8.0,1000.0,5475,"(7, 9)","(7, 9)",none,0.169460,0.261695,task5 w2v train=7-9 test=7-9 stabilized
...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,9123,"(10, 14)","(4, 6)",none,0.072821,0.459893,task5 whisper train=10-14 test=4-6
60,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5475,"(7, 9)","(4, 6)",none,0.091212,0.467246,task5 whisper train=7-9 test=4-6
61,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,20000,all,"(4, 6)",none,0.147123,0.518048,task5 whisper train=all test=4-6
62,whisper,openai/whisper-base.en,full_ft,0.000005,16.0,2000.0,5402,"(4, 6)","(4, 6)",none,0.306999,0.585561,task5 whisper train=4-6 test=4-6
